### **Objective:** Build a document-based AI assistant that answers customer questions using uploaded PDF and TXT documents.



In [1]:
!pip -q install pandas numpy pymupdf sentence-transformers faiss-cpu transformers sentencepiece accelerate

## Imports and configuration

In [2]:
import os, re, json, glob
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import pymupdf
import faiss
import torch

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
LLM_MODEL       = "google/flan-t5-large"
CHUNK_MAX_WORDS = 90
CHUNK_MIN_WORDS = 15
TOP_K           = 4
MIN_SIMILARITY  = 0.30
RELATIVE_MARGIN = 0.10
CONTEXT_CHUNKS  = 3
CURRENCY_SYMBOL = "₹"
STORE_DIR       = Path("storage/vectorstore")

print("Libraries imported successfully!")

Libraries imported successfully!


## Load the embedding model

In [3]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL)
EMBED_DIM = embedding_model.get_sentence_embedding_dimension()
print("Embedding model loaded:", EMBEDDING_MODEL, "| dimension:", EMBED_DIM)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded: sentence-transformers/all-MiniLM-L6-v2 | dimension: 384


/tmp/ipykernel_1222/51674992.py:2: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  EMBED_DIM = embedding_model.get_sentence_embedding_dimension()


## Upload documents

In [4]:
try:
    from google.colab import files
    uploaded_files = files.upload()
    document_paths = list(uploaded_files.keys())
except ImportError:
    document_paths = sorted(glob.glob("data/sample_docs/*.pdf") + glob.glob("data/sample_docs/*.txt"))

print("\nUploaded documents:")
for path in document_paths:
    print(" -", path)

Saving product_catalog.pdf to product_catalog.pdf
Saving refund_cancellation_policy.pdf to refund_cancellation_policy.pdf
Saving shipping_delivery_policy.pdf to shipping_delivery_policy.pdf
Saving technical_support_guide.pdf to technical_support_guide.pdf

Uploaded documents:
 - product_catalog.pdf
 - refund_cancellation_policy.pdf
 - shipping_delivery_policy.pdf
 - technical_support_guide.pdf


## Extract document content
Text is extracted **page by page** so every chunk can later cite its page number.

In [5]:
def extract_pdf_pages(file_path):
    pages = []
    with pymupdf.open(file_path) as pdf:
        for i, page in enumerate(pdf):
            pages.append({"page": i + 1, "text": page.get_text("text")})
    return pages

def extract_txt_pages(file_path):
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        return [{"page": 1, "text": f.read()}]

documents = []
for file_path in document_paths:
    ext = os.path.splitext(file_path)[1].lower()
    if ext == ".pdf":
        pages = extract_pdf_pages(file_path)
    elif ext == ".txt":
        pages = extract_txt_pages(file_path)
    else:
        print("Unsupported file:", file_path)
        continue
    if not any(p["text"].strip() for p in pages):
        print("No extractable text (scanned PDF?):", file_path)
        continue
    documents.append({"source": os.path.basename(file_path), "pages": pages})

print("Documents processed:", len(documents))
for doc in documents:
    n_chars = sum(len(p["text"]) for p in doc["pages"])
    print(f"\n{doc['source']}: {len(doc['pages'])} page(s), {n_chars} characters")
    print(doc["pages"][0]["text"][:300].strip())

Documents processed: 4

product_catalog.pdf: 2 page(s), 2041 characters
NorthStar Store | Fictional demonstration knowledge base
Page 1
NorthStar Store – Product Catalogue
Demo document for testing document retrieval and question answering | Version 1.0
About this catalogue
NorthStar Store is a fictional online electronics retailer created for this internship demonstrat

refund_cancellation_policy.pdf: 1 page(s), 2016 characters
NorthStar Store | Fictional demonstration knowledge base
Page 1
NorthStar Store – Refund & Cancellation Policy
Demo policy for testing retrieval | Version 1.0 | Effective 1 November 2026
Important note
This is a fictional policy created solely for the Northlanze internship demonstration. It is not a

shipping_delivery_policy.pdf: 1 page(s), 1820 characters
NorthStar Store | Fictional demonstration knowledge base
Page 1
NorthStar Store – Shipping & Delivery Policy
Demo policy for testing retrieval | Version 1.0 | Effective 1 November 2026
Important note
This is 

## Text preprocessing and chunking



In [6]:
BULLET = re.compile(r"^\s*[•●▪◦·\-\*]\s+")
SENT_SPLIT = re.compile(r"(?<=[.!?])\s+")

def norm_line(line):
    line = re.sub(r"\s+", " ", line).strip()

    line = line.replace("■", CURRENCY_SYMBOL)
    return re.sub(r"(?<![A-Za-z0-9])I(?=\d{1,3}(?:,\d{3})+\b|\d{3,}\b)", CURRENCY_SYMBOL, line)

def find_boilerplate(documents, min_pages=3, min_fraction=0.6):
    """Lines that repeat on most pages (headers/footers) are not content."""
    counts, total_pages = Counter(), sum(len(d["pages"]) for d in documents)
    for doc in documents:
        for page in doc["pages"]:
            for line in {norm_line(l).lower() for l in page["text"].splitlines() if l.strip()}:
                counts[line] += 1
    limit = max(min_pages, min_fraction * total_pages)
    return {line for line, c in counts.items() if c >= limit}

def is_noise(line, boilerplate):
    low = line.lower()
    return (low in boilerplate
            or re.fullmatch(r"(page\s*)?\d+(\s*(of|/)\s*\d+)?", low) is not None
            or " | " in line)

def is_heading(line, prev_item):
    words = line.split()
    if not (1 <= len(words) <= 9) or line[-1] in ".!?,;:":
        return False
    return prev_item is None or prev_item[-1] in ".!?:"

def parse_sections(pages, boilerplate):
    sections, cur = [], None
    for page in pages:
        for raw in page["text"].splitlines():
            line = norm_line(raw)
            if not line or is_noise(line, boilerplate):
                continue
            is_bullet = bool(BULLET.match(line))
            line = BULLET.sub("", line)
            prev = cur["items"][-1] if cur and cur["items"] else None
            if not is_bullet and is_heading(line, prev):
                cur = {"heading": line, "page": page["page"], "items": []}
                sections.append(cur)
                continue
            if cur is None:
                cur = {"heading": "Introduction", "page": page["page"], "items": []}
                sections.append(cur)
            if is_bullet or prev is None or prev[-1] in ".!?:":
                cur["items"].append(line)
            else:
                cur["items"][-1] += " " + line
    return [s for s in sections if s["items"]]

def doc_title(source):
    return Path(source).stem.replace("_", " ").replace("-", " ").title()

def build_chunks(doc, boilerplate, max_words=CHUNK_MAX_WORDS):
    title, chunks, buf = doc_title(doc["source"]), [], []

    def add(text, heading, page):
        chunks.append({"source": doc["source"], "page": page, "section": heading,
                       "text": text, "embed_text": f"{title}. {text}"})

    def flush():
        if buf:
            add(" ".join(b["text"] for b in buf), " / ".join(b["heading"] for b in buf), buf[0]["page"])
            buf.clear()

    for sec in parse_sections(doc["pages"], boilerplate):
        piece = f"{sec['heading']}: " + " ".join(sec["items"])
        n_words = len(piece.split())
        if n_words > max_words:
            flush()
            units = [s for item in sec["items"] for s in SENT_SPLIT.split(item)]
            group, count = [], 0
            for u in units:
                if group and count + len(u.split()) > max_words:
                    add(f"{sec['heading']}: " + " ".join(group), sec["heading"], sec["page"])
                    group, count = [], 0
                group.append(u); count += len(u.split())
            if group:
                add(f"{sec['heading']}: " + " ".join(group), sec["heading"], sec["page"])
            continue
        buf_words = sum(len(b["text"].split()) for b in buf)
        if buf and (buf_words >= CHUNK_MIN_WORDS or buf_words + n_words > max_words):
            flush()
        buf.append({"text": piece, "heading": sec["heading"], "page": sec["page"]})
    flush()
    return chunks

boilerplate = find_boilerplate(documents)
print("Boilerplate lines removed:", sorted(boilerplate))

all_chunks = []
for doc in documents:
    for chunk in build_chunks(doc, boilerplate):
        chunk["chunk_id"] = len(all_chunks)
        all_chunks.append(chunk)

print("\nTotal document chunks:", len(all_chunks))
display(pd.DataFrame(all_chunks)[["chunk_id", "source", "page", "section", "text"]].head(10))

Boilerplate lines removed: ['important note', 'northstar store | fictional demonstration knowledge base', 'page 1']

Total document chunks: 27


,chunk_id,source,page,section,text
0,0,product_catalog.pdf,1,About this catalogue,About this catalogue: NorthStar Store is a fic...
1,1,product_catalog.pdf,1,Laptop – NS Aero 14,Laptop – NS Aero 14: Display: 14-inch Full HD ...
2,2,product_catalog.pdf,1,Smartphone – NS Nova 5G,Smartphone – NS Nova 5G: Display: 6.5-inch OLE...
3,3,product_catalog.pdf,1,Wireless headphones – NS SoundWave,Wireless headphones – NS SoundWave: Connection...
4,4,product_catalog.pdf,1,Smartwatch – NS Pulse,Smartwatch – NS Pulse: Display: 1.8-inch colou...
5,5,product_catalog.pdf,1,Product availability and recommendations,Product availability and recommendations: Avai...
6,6,refund_cancellation_policy.pdf,1,NorthStar Store – Refund & Cancellation Policy,NorthStar Store – Refund & Cancellation Policy...
7,7,refund_cancellation_policy.pdf,1,Order cancellation,Order cancellation: A customer may request can...
8,8,refund_cancellation_policy.pdf,1,Refund eligibility,Refund eligibility: A refund may be considered...
9,9,refund_cancellation_policy.pdf,1,Refund processing time,"Refund processing time: After approval, NorthS..."


## Generate embeddings

In [7]:
chunk_texts = [c["embed_text"] for c in all_chunks]
embeddings = embedding_model.encode(chunk_texts, convert_to_numpy=True,
                                    normalize_embeddings=True, show_progress_bar=True).astype("float32")
print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding shape: (27, 384)


## Build FAISS vector database

In [8]:
# Inner product on L2-normalised vectors == cosine similarity
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
print("Total indexed chunks:", index.ntotal)

# Persist so the FastAPI service / Streamlit app can reuse the index without re-embedding
def save_vector_store(index, chunks, directory=STORE_DIR):
    directory.mkdir(parents=True, exist_ok=True)
    faiss.write_index(index, str(directory / "index.faiss"))
    (directory / "chunks.json").write_text(json.dumps(chunks, ensure_ascii=False, indent=2), encoding="utf-8")

def load_vector_store(directory=STORE_DIR):
    return (faiss.read_index(str(directory / "index.faiss")),
            json.loads((directory / "chunks.json").read_text(encoding="utf-8")))

save_vector_store(index, all_chunks)
_idx, _chunks = load_vector_store()
assert _idx.ntotal == len(_chunks) == len(all_chunks)
print("Vector store saved to", STORE_DIR, "and reloaded OK")

Total indexed chunks: 27
Vector store saved to storage/vectorstore and reloaded OK


## Retrieve relevant documents
Retrieve the top-k chunks, then **keep only those that are close to the best match** – this stops unrelated documents from reaching the LLM and from being listed as sources.

In [9]:
def retrieve_documents(query, top_k=TOP_K):
    q = embedding_model.encode([query], convert_to_numpy=True, normalize_embeddings=True).astype("float32")
    scores, ids = index.search(q, min(top_k, index.ntotal))
    return [{**all_chunks[i], "similarity": float(s)} for s, i in zip(scores[0], ids[0]) if i != -1]

def select_context(hits, min_similarity=None, margin=RELATIVE_MARGIN):
    min_similarity = MIN_SIMILARITY if min_similarity is None else min_similarity
    if not hits or hits[0]["similarity"] < min_similarity:
        return []
    best = hits[0]["similarity"]
    keep = [h for h in hits if h["similarity"] >= max(min_similarity, best - margin)]
    return keep[:CONTEXT_CHUNKS]

# Test retrieval
query = "How long does a refund take?"
for hit in retrieve_documents(query):
    print(f"\n{hit['source']} | page {hit['page']} | {hit['section']} | similarity {hit['similarity']:.3f}")
    print(hit["text"][:300])
    print("-" * 60)


refund_cancellation_policy.pdf | page 1 | Refund processing time | similarity 0.627
Refund processing time: After approval, NorthStar Store aims to initiate the refund within 5–7 business days. After initiation, the time for the amount to appear depends on the bank, card issuer, or payment provider. The original payment method is normally used for the refund where supported.
------------------------------------------------------------

refund_cancellation_policy.pdf | page 1 | Refund eligibility | similarity 0.415
Refund eligibility: A refund may be considered for an eligible cancelled order, a damaged item, an incorrect item, or an approved return. The item and request may be subject to inspection and eligibility checks. Items marked as non-returnable on their product page may not qualify for a standard retu
------------------------------------------------------------

refund_cancellation_policy.pdf | page 1 | Information not covered | similarity 0.412
Information not covered: This d

## Load the language model

In [10]:
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
llm_model = AutoModelForSeq2SeqLM.from_pretrained(LLM_MODEL)
device = "cuda" if torch.cuda.is_available() else "cpu"
llm_model = llm_model.to(device).eval()

def llm_generate(prompt, max_new_tokens=128):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(device)
    with torch.no_grad():
        output = llm_model.generate(**inputs, max_new_tokens=max_new_tokens, num_beams=4, do_sample=False)
    return tokenizer.decode(output[0], skip_special_tokens=True).strip()

print(f"{LLM_MODEL} loaded on {device}")

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

... (2 outputs left out)

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

google/flan-t5-large loaded on cpu


## Generate RAG responses
Two safeguards against inventing answers: (1) the similarity threshold, and (2) the model is told to reply `unanswerable` when the context lacks the answer; that reply is converted into a clean "not found" message **with no sources attached**.

In [11]:
NO_ANSWER_MESSAGE = ("I could not find this information in the available documents. "
                     "Please contact customer support for help.")

REFUSAL_STARTS = ("unanswerable", "i don't know", "i do not know", "the information is not",
                  "information is not available", "not mentioned", "no information", "not specified in the")

NOT_SPECIFIED = re.compile(r"\b(does not|doesn't|do not|did not)\s+(specify|provide|define|contain|include|mention|cover)\b"
                           r"|\bnot\s+(specified|provided|defined|mentioned|covered)\b", re.I)

def is_refusal(answer):
    a = answer.lower().strip(" .\n")
    return len(a) < 2 or a.startswith(REFUSAL_STARTS)

def says_not_specified(answer):
    first_sentence = SENT_SPLIT.split(answer.strip())[0]
    return bool(NOT_SPECIFIED.search(first_sentence))

def build_prompt(query, context_hits):
    context = "\n\n".join(f"[{h['source']}, page {h['page']}] {h['text']}" for h in context_hits)
    return ("Answer the question using only the context below. Give a short, complete answer.\n"
            "If the context does not contain the answer, reply with exactly: unanswerable\n\n"
            f"Context:\n{context}\n\nQuestion: {query}\nAnswer:")

def generate_rag_response(query, top_k=TOP_K):
    """status: 'answered' | 'not_specified' (the documents explicitly say so) | 'not_in_documents'"""
    hits = retrieve_documents(query, top_k)
    top_score = round(hits[0]["similarity"], 4) if hits else 0.0
    context_hits = select_context(hits)

    if not context_hits:
        return {"status": "not_in_documents", "answer": NO_ANSWER_MESSAGE, "answerable": False,
                "top_score": top_score, "sources": []}

    answer = llm_generate(build_prompt(query, context_hits))
    if is_refusal(answer):
        return {"status": "not_in_documents", "answer": NO_ANSWER_MESSAGE, "answerable": False,
                "top_score": top_score, "sources": []}

    sources = [{"document": h["source"], "page": h["page"], "section": h["section"],
                "similarity": round(h["similarity"], 4)} for h in context_hits]
    if says_not_specified(answer):
        return {"status": "not_specified", "answerable": False, "top_score": top_score, "sources": sources,
                "answer": answer.rstrip(". ") + ". Please contact customer support for details."}
    return {"status": "answered", "answer": answer, "answerable": True, "top_score": top_score, "sources": sources}

def show(question, result):
    print("\nCUSTOMER QUESTION:", question)
    print("AI RESPONSE      :", result["answer"])
    print("STATUS           :", result["status"], f"(top similarity {result['top_score']})")
    for s in result["sources"]:
        print("  source ->", s)
    print("=" * 80)

## Test the RAG system
In-scope questions (each with the document that *should* be retrieved) and out-of-scope questions (which must be declined).

In [12]:
in_scope = [
    ("What are the specifications of the NS Aero 14?",                "product_catalog.pdf"),
    ("What is the price of the NS Nova 5G?",                          "product_catalog.pdf"),
    ("Does the NS Pulse smartwatch work with iPhone?",                "product_catalog.pdf"),
    ("What is included in the box with the NS SoundWave?",            "product_catalog.pdf"),
    ("How long does an approved refund take?",                        "refund_cancellation_policy.pdf"),
    ("Can I cancel my order after it has been shipped?",              "refund_cancellation_policy.pdf"),
    ("How do I request a refund for a damaged item?",                 "refund_cancellation_policy.pdf"),
    ("How long does delivery take in metro locations?",               "shipping_delivery_policy.pdf"),
    ("Can I change my delivery address after dispatch?",              "shipping_delivery_policy.pdf"),
    ("My tracking has not updated for three days, what should I do?", "shipping_delivery_policy.pdf"),
    ("What should I do if my password reset email does not arrive?",  "technical_support_guide.pdf"),
    ("My money was debited but I got no order confirmation",          "technical_support_guide.pdf"),
]

not_specified = [
    ("Does the policy specify a cancellation fee?",   "refund_cancellation_policy.pdf"),
    ("What is the return window?",                    "refund_cancellation_policy.pdf"),
    ("What is the restocking fee?",                   "refund_cancellation_policy.pdf"),
    ("Is there a free shipping threshold?",           "shipping_delivery_policy.pdf"),
    ("Is the NS Nova 5G in stock right now?",         "product_catalog.pdf"),
]
out_of_scope = [
    "Who is the CEO of Apple?",
    "What is the weather in Mumbai today?",
    "Can you suggest a good pasta recipe?",
    "What is the capital of Australia?",
]

for question, _ in in_scope + not_specified + [(q, None) for q in out_of_scope]:
    show(question, generate_rag_response(question))


CUSTOMER QUESTION: What are the specifications of the NS Aero 14?
AI RESPONSE      : 14-inch Full HD display. Memory and storage: 16 GB RAM and 512 GB SSD. Connectivity: Wi-Fi 6 and Bluetooth 5.2. Ports: two USB-A ports, one USB-C port, and an HDMI port
STATUS           : answered (top similarity 0.6062)
  source -> {'document': 'product_catalog.pdf', 'page': 1, 'section': 'Laptop – NS Aero 14', 'similarity': 0.6062}

CUSTOMER QUESTION: What is the price of the NS Nova 5G?
AI RESPONSE      : 24,999
STATUS           : answered (top similarity 0.7518)
  source -> {'document': 'product_catalog.pdf', 'page': 1, 'section': 'Smartphone – NS Nova 5G', 'similarity': 0.7518}

CUSTOMER QUESTION: Does the NS Pulse smartwatch work with iPhone?
AI RESPONSE      : works with supported Android and iOS devices using the companion app
STATUS           : answered (top similarity 0.643)
  source -> {'document': 'product_catalog.pdf', 'page': 1, 'section': 'Smartwatch – NS Pulse', 'similarity': 0.643}

C

## Evaluate retrieval and calibrate the threshold


In [13]:
rows = []
for question, expected in in_scope + not_specified:
    hits = retrieve_documents(question)
    rows.append({"question": question, "expected": expected, "top1": hits[0]["source"],
                 "hit@1": hits[0]["source"] == expected,
                 "hit@k": expected in [h["source"] for h in hits],
                 "top_similarity": round(hits[0]["similarity"], 3)})
retrieval_df = pd.DataFrame(rows)
display(retrieval_df)
print(f"Retrieval Hit@1: {retrieval_df['hit@1'].mean():.0%}   Hit@{TOP_K}: {retrieval_df['hit@k'].mean():.0%}")

oos_scores = [retrieve_documents(q)[0]["similarity"] for q in out_of_scope]
in_scores  = retrieval_df["top_similarity"].tolist()   # all of these topics ARE in the documents
print(f"\nIn-scope top-1 similarity : min {min(in_scores):.3f}, mean {np.mean(in_scores):.3f}")
print(f"Out-of-scope top-1 similarity: max {max(oos_scores):.3f}, mean {np.mean(oos_scores):.3f}")
if min(in_scores) > max(oos_scores):
    print(f"-> Clean gap. A threshold around {(min(in_scores) + max(oos_scores)) / 2:.2f} separates the two groups.")
else:
    print("-> Scores overlap, so no single threshold is perfect; rely on the 'unanswerable' check as the second guard.")
results_oos = [generate_rag_response(q) for q in out_of_scope]
print(f"Out-of-scope questions declined: {sum(r['status'] == 'not_in_documents' for r in results_oos)}/{len(out_of_scope)}")
results_ns = [generate_rag_response(q) for q, _ in not_specified]
print(f"'Not specified' questions without an invented answer: "
      f"{sum(r['status'] in ('not_specified', 'not_in_documents') for r in results_ns)}/{len(not_specified)}")
print("(Also read those answers above: a good response says the documents do not provide the information.)")

,question,expected,top1,hit@1,hit@k,top_similarity
0,What are the specifications of the NS Aero 14?,product_catalog.pdf,product_catalog.pdf,True,True,0.606
1,What is the price of the NS Nova 5G?,product_catalog.pdf,product_catalog.pdf,True,True,0.752
2,Does the NS Pulse smartwatch work with iPhone?,product_catalog.pdf,product_catalog.pdf,True,True,0.643
3,What is included in the box with the NS SoundW...,product_catalog.pdf,product_catalog.pdf,True,True,0.626
4,How long does an approved refund take?,refund_cancellation_policy.pdf,refund_cancellation_policy.pdf,True,True,0.655
5,Can I cancel my order after it has been shipped?,refund_cancellation_policy.pdf,refund_cancellation_policy.pdf,True,True,0.669
6,How do I request a refund for a damaged item?,refund_cancellation_policy.pdf,refund_cancellation_policy.pdf,True,True,0.610
7,How long does delivery take in metro locations?,shipping_delivery_policy.pdf,shipping_delivery_policy.pdf,True,True,0.782
8,Can I change my delivery address after dispatch?,shipping_delivery_policy.pdf,shipping_delivery_policy.pdf,True,True,0.778
9,"My tracking has not updated for three days, wh...",shipping_delivery_policy.pdf,shipping_delivery_policy.pdf,True,True,0.588


Retrieval Hit@1: 100%   Hit@4: 100%

In-scope top-1 similarity : min 0.379, mean 0.643
Out-of-scope top-1 similarity: max 0.254, mean 0.148
-> Clean gap. A threshold around 0.32 separates the two groups.
Out-of-scope questions declined: 4/4
'Not specified' questions without an invented answer: 4/5
(Also read those answers above: a good response says the documents do not provide the information.)
